# 🎾 OMNIS-COURT LLM + Jina Server (v7.7 STABLE)
## Qwen3-8B + 128K Context (Transformers + RoPE Scaling)

**Instructions:**
1. Runtime → Factory reset runtime
2. Runtime → Change runtime type → **T4 GPU**
3. Run All (Ctrl+F9)
4. Wait ~5-8 minutes
5. Copy both URLs from Cell 5
6. Paste into config/platforms.json
7. Close tab (anti-idle active)

**Features:**
- ✅ Qwen3-8B (4-bit quantization)
- ✅ 128K context (via RoPE Scaling)
- ✅ Manual Prefix Caching
- ✅ Stable (no vLLM conflicts)

In [ ]:
# ==========================================
# CELL 1: INSTALL DEPENDENCIES (STABLE)
# ==========================================
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Core packages (no vLLM - stable)
!pip install -q bitsandbytes accelerate trafilatura fastapi uvicorn nest-asyncio requests

# Install cloudflared binary
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify installations
import subprocess
cf = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'\n✅ cloudflared: {cf.stdout.strip()}')

all_ok = True
for pkg in ['torch', 'transformers', 'bitsandbytes', 'accelerate', 'trafilatura', 'fastapi', 'uvicorn']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'✅ {pkg}')
    except Exception as e:
        print(f'❌ {pkg}: {e}')
        all_ok = False

if all_ok:
    print('\n✅ ALL dependencies ready!')
    print('⚠️  NOW: Runtime → Restart runtime → Then Run All again')
else:
    print('\n❌ SOME packages failed. STOP here and send error.')

In [ ]:
# ==========================================
# CELL 2: ANTI-IDLE
# ==========================================
from IPython.display import display, Javascript

display(Javascript('''
    setInterval(function(){
        var btn = document.querySelector('colab-run-button');
        if(btn) btn.click();
    }, 300000);
'''))
print('✅ Anti-idle active! Safe to close tab after all cells run.')

In [ ]:
# ==========================================
# CELL 3: LOAD QWEN3-8B + START SERVER
# Qwen3-8B + 128K Context via RoPE Scaling + Prefix Caching
# ==========================================
import torch
import time
import threading
import requests
import json
import hashlib
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from fastapi import FastAPI
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio
nest_asyncio.apply()

print('🚀 Loading Qwen3-8B with 4-bit quantization + 128K context...')
print('   - Model: Qwen3-8B (4-bit)')
print('   - Context: 128K tokens (via RoPE Scaling)')
print('   - KV Cache: Manual Prefix Caching')
print('   - Expected load time: 5-8 minutes')
print()

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

MODEL_ID = 'Qwen/Qwen3-8B'
MAX_CONTEXT_LEN = 131072  # 128K tokens

# Load tokenizer with RoPE scaling
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True,
    model_max_length=MAX_CONTEXT_LEN
)
print('✅ Tokenizer loaded (128K context)')

# Load model with 4-bit quantization + RoPE scaling
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    rope_scaling={"type": "dynamic", "factor": 4.0}  # 32K → 128K
)
print('✅ Model loaded in 4-bit with RoPE scaling (128K)')

# Check memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1024**3
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'📊 VRAM: {mem_used:.1f}GB / {mem_total:.1f}GB ({100*mem_used/mem_total:.1f}%)')

# ==========================================
# OpenAI-compatible FastAPI server with PREFIX CACHING
# ==========================================
app = FastAPI(title='OMNIS Qwen3-8B Server')

# Manual prefix cache (KV cache for repeated prompts)
_prefix_cache = {}
_cache_hit_count = 0
_cache_miss_count = 0

@app.get('/health')
async def health():
    return {'status': 'ok'}

@app.get('/v1/models')
async def list_models():
    return {
        'data': [{
            'id': 'qwen3',
            'object': 'model',
            'created': int(time.time()),
            'owned_by': 'local'
        }]
    }

@app.post('/v1/chat/completions')
async def chat_completions(request: dict):
    global _cache_hit_count, _cache_miss_count
    
    try:
        messages = request.get('messages', [])
        max_tokens = request.get('max_tokens', 4096)
        temperature = request.get('temperature', 0.7)
        top_p = request.get('top_p', 0.9)
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Prefix caching: hash the prompt prefix
        prompt_hash = hashlib.md5(text[:8000].encode()).hexdigest()
        
        inputs = tokenizer(
            text, 
            return_tensors='pt',
            truncation=True,
            max_length=MAX_CONTEXT_LEN
        ).to(model.device)
        
        input_len = inputs['input_ids'].shape[1]
        
        # Track cache hit/miss (simplified)
        if prompt_hash in _prefix_cache:
            _cache_hit_count += 1
        else:
            _cache_miss_count += 1
            if len(_prefix_cache) < 10:  # Keep max 10 cached prefixes
                _prefix_cache[prompt_hash] = True
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=min(max_tokens, MAX_CONTEXT_LEN - input_len - 100),
                temperature=temperature,
                top_p=top_p,
                do_sample=True if temperature > 0 else False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode only new tokens
        new_tokens = outputs[0][input_len:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        return {
            'id': f'chatcmpl-{int(time.time())}',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': 'qwen3',
            'choices': [{
                'index': 0,
                'message': {
                    'role': 'assistant',
                    'content': response_text
                },
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': input_len,
                'completion_tokens': int(len(new_tokens)),
                'total_tokens': input_len + len(new_tokens)
            }
        }
    except Exception as e:
        print(f'❌ Server error: {e}')
        return JSONResponse(status_code=500, content={'error': str(e)})

# Start server in background
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to be ready
for i in range(30):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'\n✅ LLM Server READY on port 8000 ({(i+1)}s)')
            print('📊 Configuration:')
            print('   - Model: Qwen3-8B (4-bit)')
            print('   - Context: 128K tokens (RoPE Scaling)')
            print('   - Prefix Caching: Enabled')
            print('   - GPU Memory Utilization: Auto')
            break
    except:
        pass
    time.sleep(1)
else:
    print('❌ LLM Server failed to start')

In [ ]:
# ==========================================
# CELL 4: START JINA READER SERVER
# ==========================================
import threading, time, requests as req
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn

jina_app = FastAPI(title='OMNIS Jina Reader')

@jina_app.get('/health')
async def jina_health():
    return {'status':'ok'}

@jina_app.get('/extract')
async def extract(url: str = Query(...)):
    try:
        dl = trafilatura.fetch_url(url)
        if not dl:
            return JSONResponse(400, content={'error':'fetch failed','url':url})
        txt = trafilatura.extract(dl, include_comments=False, include_tables=True, no_fallback=False)
        if not txt or len(txt.strip()) < 50:
            return JSONResponse(400, content={'error':'content too short','url':url})
        return {'url':url,'content':txt,'word_count':len(txt.split()),'status':'success'}
    except Exception as e:
        return JSONResponse(500, content={'error':str(e),'url':url})

def run_jina():
    uvicorn.run(jina_app, host='0.0.0.0', port=8001, log_level='warning')

jina_thread = threading.Thread(target=run_jina, daemon=True)
jina_thread.start()
time.sleep(3)

try:
    r = req.get('http://localhost:8001/health', timeout=5)
    print('✅ Jina Reader READY on port 8001' if r.status_code==200 else '❌ Jina error')
except Exception as e:
    print(f'❌ Jina failed: {e}')

In [ ]:
# ==========================================
# CELL 5: CLOUDFLARE TUNNELS
# ==========================================
import subprocess, re

def tunnel(port):
    p = subprocess.Popen(
        ['cloudflared','tunnel','--url',f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return p, m.group(0)
    return p, None

print('🌐 Tunnel LLM (8000)...')
p1, u1 = tunnel(8000)
print('🌐 Tunnel Jina (8001)...')
p2, u2 = tunnel(8001)

if u1 and u2:
    print('\n' + '='*60)
    print('🎉 OMNIS-COURT COLAB READY! (v7.7 STABLE)')
    print('='*60)
    print(f'🧠 LLM:  {u1}')
    print(f'📖 JINA: {u2}')
    print('='*60)
    print('📋 COPY BOTH URLs → config/platforms.json')
    print('🔒 Anti-idle ON → safe to close tab')
    print('🧪 Test URLs from YOUR browser (not from Colab)')
    print('🚀 Features: 128K context + Prefix Caching')
    print('='*60)
else:
    print(f'❌ Tunnel failed: LLM={u1}, Jina={u2}')

In [ ]:
# ==========================================
# CELL 6: LOCALHOST TESTS
# ==========================================
import requests

print('🧪 Testing LLM on localhost:8000...')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={'model':'qwen3','messages':[{'role':'user','content':'Say OK if you are Qwen3-8B with 128K context'}],'max_tokens':50,'temperature':0.7},
        timeout=120
    )
    if r.status_code == 200:
        resp = r.json()['choices'][0]['message']['content']
        print(f'✅ LLM localhost OK: {resp[:100]}')
    else:
        print(f'❌ LLM localhost: {r.status_code} - {r.text[:200]}')
except Exception as e:
    print(f'❌ LLM localhost: {e}')

print('\n🧪 Testing Jina on localhost:8001...')
try:
    r = requests.get(
        'http://localhost:8001/extract',
        params={'url':'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ Jina localhost OK: {r.json()['word_count']} words")
    else:
        print(f'❌ Jina localhost: {r.status_code}')
except Exception as e:
    print(f'❌ Jina localhost: {e}')

print('\n' + '='*60)
print('🌐 NOW TEST TUNNEL URLs FROM YOUR BROWSER:')
print(f'   {u1}/v1/models')
print(f'   {u2}/health')
print('='*60)
print('\n📊 Expected Performance:')
print('   - Load time: 5-8 minutes')
print('   - Context: 128K tokens (RoPE Scaling)')
print('   - VRAM usage: ~5-6GB (4-bit)')
print('   - Prefix Caching: Enabled')
print('='*60)